In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.text_cell_render.rendered_html{font size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input{font-family:Consolas; font-size:12pt;}
div.prompt {min width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe {font-size:12px;}
</style>
"""))

# [RAG 절차]

1. 문서를 읽는다
    - %pip install --upgrade --quiet docx2txt
2. 문서를 쪼갠다
    - %pip install -qU langchain-text-splitters
3. 쪼갠 문서를 임베딩하여 vector database에 넣음 (local에 저장) cf. 클라우드에 저장
    - %pip install -q langchain-chroma
4. 질문을 이용해 유사도 검색
5. 유사도 검색한 문서를 LLM에 질문과 함께 전달하여 답변 얻음 (랭체인 사용 가능)
    - %pip install -q langchain
    - (https://smith.langchain.com)에서 key 생성 후 .env에 LANGCHAIN_API_KEY로 추가

# 0. 패키지 설치

In [2]:
# 텍스트를 chunk로 나누는 기능만 있는 경량 모듈
%pip install -qU langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [3]:
# 벡터DB (로컬 DB) / chromadb와 다름
%pip install -q langchain-chroma

Note: you may need to restart the kernel to use updated packages.


In [4]:
#Langchain 사용
%pip install -q langchain

Note: you may need to restart the kernel to use updated packages.


# 1. 문서읽기(X)

In [2]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('data/소득세법(법률)(제21065호)(20260102).docx')
document = loader.load()

CPU times: total: 3.8 s
Wall time: 4.09 s


In [3]:
len(document)

1

# 2. 문서를 쪼개면서 읽기(O)
- https://docs.langchain.com/oss/python/integrations/splitters

## 2.1 1500토큰씩 쪼개서 읽어오기

In [4]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import TokenTextSplitter
loader = Docx2txtLoader('data/소득세법(법률)(제21065호)(20260102).docx')
# gpt-4, gpt-4o, gpt-4 turbo, gpt-4o-mini, embedding 모델들은 다 같은 방식으로 토큰 추출
text_splitter = TokenTextSplitter(
    encoding_name='cl100k_base', # 토큰을 세는 방식 이름
    chunk_size=1500, # chunk 당 토큰 수 기준
    chunk_overlap=200, # chunk 당 오버랩 토큰 수
)
documents = loader.load_and_split(text_splitter=text_splitter)

CPU times: total: 3.78 s
Wall time: 5.65 s


In [6]:
len(documents) # chunk 수

180

In [10]:
# chunk 글자수
print([len(document.page_content) for document in documents])

[1699, 1656, 1641, 1650, 1738, 1442, 1287, 1535, 1325, 1619, 1596, 1588, 1566, 1639, 1622, 1559, 1612, 1638, 1573, 1465, 1436, 1609, 1456, 1497, 1635, 1606, 1533, 1649, 1662, 1595, 1603, 1678, 1595, 1637, 1601, 1539, 1561, 1594, 1693, 1708, 1657, 1627, 1636, 1659, 1667, 1595, 1491, 1485, 1645, 1709, 1629, 1617, 1495, 1626, 1612, 1620, 1609, 1576, 1636, 1602, 1556, 1563, 1600, 1616, 1643, 1691, 1635, 1685, 1621, 1631, 1609, 1605, 1603, 1604, 1698, 1686, 1702, 1612, 1539, 1558, 1651, 2060, 1562, 1606, 1557, 1648, 1594, 1615, 1766, 1651, 1690, 1576, 1536, 1553, 1638, 1685, 1693, 1694, 1664, 1529, 1627, 1703, 1675, 1546, 1585, 1687, 1679, 1714, 1603, 1655, 1648, 1495, 1531, 1562, 1594, 1646, 1543, 1449, 1593, 1559, 1521, 1473, 1519, 1545, 1668, 1700, 1692, 1655, 1648, 1741, 1670, 1628, 1639, 1623, 1638, 1642, 1666, 1658, 1594, 1591, 1561, 1641, 1498, 1610, 1567, 1613, 1636, 1619, 1531, 1496, 1702, 1598, 1579, 1627, 1559, 1585, 1665, 1565, 1616, 1564, 1612, 1535, 1512, 1557, 1576, 1628, 165

In [12]:
# chunk 글자수 최대값, 최소값
print(max([len(document.page_content) for document in documents]))
print(min([len(document.page_content) for document in documents[:-1]]))

2060
1287


## 2.2 1500글자 쪼개서 읽어오기

In [1]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('data/소득세법(법률)(제21065호)(20260102).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, #문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=200,
    #separators=['\n\n', '\n', ' ', '']
)
# 재귀적으로 다음 순서대로 시도:
# 1. \n\n (문단구분)
# 2. \n (줄바꿈)
# 3. " " (공백)
# 4. "" (최후에는 글자 단위로 chunking)
documents = loader.load_and_split(text_splitter=text_splitter)
print('chunk 갯수 :', len(documents))

chunk 갯수 : 193
CPU times: total: 3.77 s
Wall time: 3.9 s


In [2]:
print(max([len(document.page_content) for document in documents]))
print(min([len(document.page_content) for document in documents]))


1496
325


# 3. 쪼갠 문서를 임베딩 -> 벡터 데이터베이스 저장
- 임베딩 모델 : upstage의 solar-embedding-1-large-passage
- 벡터데이터베이스(벡터 store) : chroma

In [2]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(model='solar-embedding-1-large-passage')

In [3]:
%%time
from langchain_chroma import Chroma
# 데이터 처음 저장할 때
# database = Chroma.from_documents(
#     documents=documents, # chunk
#     embedding=embedding, # 임베딩 객체
#     collection_name='tax-collection', # db의 table과 비슷한 느낌, 생략시 랜덤
#     persist_directory='./chroma_upstage', # 생략시 로컬DB에 저장 안 됨. 프로그램 종료 시 DB 날라감
# )

CPU times: total: 4.03 s
Wall time: 35.4 s


In [15]:
# 이미 저장된 vector DB(store)를 사용할 때
database = Chroma(
    embedding_function=embedding,
    persist_directory='./chroma_upstage',
    collection_name='tax-collection',
)

In [24]:
results = database._collection.get(include=['embeddings', 'documents', 'metadatas'])
print("데이터 수 :", len(results['ids']))
print("문서 임베딩 차원 수 :", len(results['embeddings'][0]))
print("1번째 chunk의 임베딩 샘플 :", results['embeddings'][1])
print('1번째 chunk의 원본 :', results['documents'][1][:50])
print('1번째 chunk의 metadata :', results['metadatas'][1])

데이터 수 : 180
문서 임베딩 차원 수 : 3072
1번째 chunk의 임베딩 샘플 : [ 0.01991478 -0.01470464 -0.00057961 ...  0.0058937  -0.03365059
 -0.00657188]
1번째 chunk의 원본 : . 구성원 간 이익의 분배비율이 정하여져 있지 아니하나 사실상 구성원별로 이익이 분배되는 
1번째 chunk의 metadata : {'source': 'data/소득세법(법률)(제21065호)(20260102).docx'}


# 4. vectorDB에 질문과 유사도 검색(답변 생성을 위한 retrieval)

In [25]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
retrieved_docs = database.similarity_search(query=query, k=2) # 기본 k값은 4

In [32]:
retrieved_doc = "\n\n---\n\n".join([retrieved_doc.page_content for retrieved_doc in retrieved_docs])

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM에 전달 (1)

In [34]:
from langchain_openai import ChatOpenAI
load_dotenv()
llm = ChatOpenAI(model='gpt-5-nano')

In [35]:
prompt = f'''[identity]
- 당신은 최고의 한국 소득세 전문가입니다
- [context]를 참고해서 사용자의 질문에 답변해주세요
[context]는 다음과 같아요
{retrieved_doc}
질문 : {query}'''

In [37]:
ai_message = llm.invoke(prompt)

In [38]:
ai_message.usage_metadata

{'input_tokens': 2130,
 'output_tokens': 5703,
 'total_tokens': 7833,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 4928}}

In [39]:
print(ai_message.content)

답변 요점
- exact 금액은 개인 공제·세액공제 여부에 따라 달라지지만, 일반적인 가정 아래 연봉 5,000만 원인 직장인의 경우 대략적으로 국세 소득세 약 4.7~5.3백만 원, 여기에 지방소득세 10%가 추가되어 총 약 5.2~5.8백만 원 정도의 세금이 부담될 가능성이 큽니다.

근로소득 세액의 아주 간단한 예시 계산(가정: 단독, 자녀 등 추가 공제 없음)
- 연간 총소득: 50,000,000원
- 근로소득공제(근로소득에 대한 기본 공제): 9,000,000원 가정 예시(근로소득공제 금액은 연봉 수준에 따라 다르므로 실제와 차이 있음)
- 기본공제: 1,500,000원
- 과세표준(세율 적용 기준 소득): 50,000,000 - 9,000,000 - 1,500,000 = 약 39,500,000원

국세(근로소득에 적용되는 누진세율 예시, 표준 구간)
- 0 ~ 12,000,000원: 6% → 720,000원
- 12,000,001 ~ 46,000,000원: 15% → (39,500,000 - 12,000,000) 27,500,000원 × 15% = 4,125,000원
- 합계 국세: 약 4,845,000원

지방소득세
- 국세의 10% 가산: 약 484,500원

총합(국세+지방세): 약 5,329,500원

다른 근로소득공제가 더 크거나(예: 12,000,000원 등) 기본공제 상황이 다르면 결과는 달라집니다.
- 예를 들어 근로소득공제가 12,000,000원인 경우 과세표준이 약 38,500,000원이 되고 국세는 약 4,695,000원, 지방세 ≈ 469,500원으로 합계 약 5,165,000원이 됩니다.

주의점
- 위 수치는 매우 일반적인 가정에 따른 근사치입니다. 실제 금액은:
  - 근로소득공제 exact 금액(연봉 구간별 공식 표에 따른 계산)
  - 기본공제(본인, 배우자, 자녀 등 가족공제)
  - 특별공제(보험료, 의료비, 교육비 등)
  - 소득공제 및 세액공제 항목(연금보험료, 건강보험료 등)
  에 따라 크게 달라집니다.
-

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM에 전달 (2)

In [45]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-5-nano')

promptTemplate = ChatPromptTemplate([
    ('system', '당신은 최고의 한국 소득세 전문가입니다'),
    ('human', f'''다음 문맥을 참고하여 질문에 답변하세요.
    답을 모르면 모른다고 말하세요. 최대 3문장으로 간결하게 답변하세요.
    질문 : {{question}}
    문맥 : {{context}}
    답변 :''')
])
promptTemplate

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 최고의 한국 소득세 전문가입니다'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='다음 문맥을 참고하여 질문에 답변하세요.\n    답을 모르면 모른다고 말하세요. 최대 3문장으로 간결하게 답변하세요.\n    질문 : {question}\n    문맥 : {context}\n    답변 :'), additional_kwargs={})])

In [46]:
prompt = promptTemplate.invoke({'context':retrieved_doc, 'question':query})
prompt

ChatPromptValue(messages=[SystemMessage(content='당신은 최고의 한국 소득세 전문가입니다', additional_kwargs={}, response_metadata={}), HumanMessage(content='다음 문맥을 참고하여 질문에 답변하세요.\n    답을 모르면 모른다고 말하세요. 최대 3문장으로 간결하게 답변하세요.\n    질문 : 연봉 5천만원인 직장인의 소득세는 얼마인가요?\n    문맥 : 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31., 2024. 12. 31., 2025. 10. 1., 2025. 12. 23.>\n\n1. 「공익신탁법」에 따른 공익신탁의 이익\n\n2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득\n\n가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득\n\n나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다). 이 경우 주택 수의 계산 및 주택임대소득의 산정 등 필요한 사항은 대통령령으로 정한다.\n\n다. 대통령령으로 정하는 농어가부업소득\n\n라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득\n\n마. 조림기간 5년 이상인 임지(林地)의 임목(林木)의 벌채 또는 양도로 발생하는 소득으로서 연 3천만원 이하의 금액. 이 경우 조림기간 및 세액의 계산 등 필요한 사항은 대통령령으로 정한다.\n\n바. 대통령령으로 정하는 작물재배업에서 발생하는 소득\n\n사. 대통령령으로 정하는 어로어업 또는 양식어업에서 발생하는 소득\n\n3. 근로소득과 퇴직소득 중 다음 각 목의 어느 하나에 해당하는 소득\n\n가. 대통령령으로 정하는 복무 중인 병(兵)이 받는 급

In [47]:
llm.invoke(prompt)

AIMessage(content='주어진 문맥만으로는 정확한 소득세 금액을 산정할 수 없습니다. 연봉 5천만 원의 소득세는 근로소득공제, 인적공제 등 공제항목과 부양가족 여부에 따라 달라지며 누진세율이 적용됩니다. 정확한 금액을 원하시면 공제 내역을 알려주시면 계산해 드리겠습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1829, 'prompt_tokens': 2152, 'total_tokens': 3981, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1728, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CvasMFXr4jmpUsF5g8TQHeWpgTmC9', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b9b9f-81df-7ca2-9574-979e3ddc3333-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2152, 'output_tokens': 1829, 'total_tokens': 3981, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoni

In [49]:
# 위의 예제를 한번에 답변만 출력
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()
# output_parser.invoke(llm.invoke(promptTemplate.invoke({'context':retrieved_doc, 'question':query})))

# 6. Langchain으로 답변 생성

In [51]:
# 위의 예제를 Langchain으로 답변 생성
rag_chain = promptTemplate | llm | output_parser
rag_chain.invoke({'context':retrieved_doc, 'question':query})

'이 문맥에는 구체적인 세율표나 공제액이 제시되지 않아 연봉 5천만원의 정확한 소득세를 산출할 수 없습니다.\n근로소득세는 공제와 누진세율이 개인 상황에 따라 달라지므로, 정확한 금액은 본인 상황에 맞춘 계산이 필요합니다.\n국세청 연말정산/홈택스의 계산기를 이용해 확인하시길 권합니다.'

## Langchain 전달
- smith.langchain.com에서 key 생성 후 .env에 LANGCHAIN_API_KEY 추가

In [4]:
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

# 1. LLM과 임베딩 초기화
load_dotenv()
llm = ChatUpstage(model='solar-pro2-251215')
embedding = UpstageEmbeddings(model='solar-embedding-1-large-passage')
# 2. vector store load
vectorstore = Chroma(
    embedding_function=embedding,
    collection_name='tax-collection',
    persist_directory='./chroma_upstage'
)
# 3. Retriever 생성
retriever = vectorstore.as_retriever(
    search_type='similarity', 
    search_kwargs={'k':4}
)
# 4. 프롬프트 템플릿
template = f'''당신은 최고의 한국 소득세 전문가입니다.
다음 문맥을 참고하여 질문에 답하세요.
답을 모르면 모른다고 답하세요.
최대 3문장으로 간결하게 답변하세요.
질문:{{query}}
문맥:{{context}}
답변:'''
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 document를 텍스트로 변환하는 함수
def format_documents(documents):
    return "\n\n---\n\n".join([retrieved_doc.page_content for retrieved_doc in documents])

In [5]:
# 6. RAG 체인 구성 (LCEL 방식)
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {
        'context':retriever | format_documents,
        'query':RunnablePassthrough() # 질문 그대로 전달
    }
    | prompt # prompt에 context와 query 주입
    | llm # llm에 prompt 주입
    | StrOutputParser() # StrOutputParser에 llm 주입
)

# 7. 실행
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
rag_chain.invoke(query)

'제공된 문맥에는 연봉 5천만원에 대한 구체적인 소득세 계산 방법이 명시되어 있지 않습니다. 따라서 정확한 세액 계산은 불가능합니다. 다만, 근로소득공제 및 세액공제(제59조) 규정을 적용할 수 있으나 추가 정보(가족 구성원, 출산 여부 등)가 필요합니다. 정확한 계산을 위해서는 국세청 홈텍스 또는 전문가 상담이 필요합니다.'